<center><h1>Introduction to Transformers from Scratch</h1></center>

## 📌 Task & Notebook Overview
Welcome to the Transformer Lab! In this notebook, you will build the core components of the **Transformer Architecture** (*Attention Is All You Need*, Vaswani et al., 2017) step-by-step using **PyTorch**.

- **Goal:** Understand how **Self-Attention** and **Multi-Head Attention** compute relationships between words in a sequence without using recurrence (RNNs/LSTMs).
- **Task:** Sequence Classification (Sentiment Analysis on IMDB / Sequence Toy Task).
- **Input:** Tokenized Sequence `[batch_size, seq_len]`
- **Output:** Predicted Class Sentiment (`0` or `1`)

---
### 💡 Why Transformers over RNNs/LSTMs?
1. **Parallel Computing:** Unlike RNNs which compute word-by-word sequentially ($h_t$ depends on $h_{t-1}$), Transformers compute attention across **all tokens at once** in parallel!
2. **Direct Long-Range Connections:** Any token can attend to any other token directly in a single layer, eliminating the vanishing gradient problem over long sentences.


In [ ]:
# Import required libraries
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set random seed for reproducibility
torch.manual_seed(42)
print("PyTorch version:", torch.__version__)


## 1. Positional Encoding 📍

Since Transformers process all input tokens in parallel without recurrence or convolution, **they have no inherent sense of word order or position**.

To fix this, we add a **Positional Encoding (PE)** vector to each token embedding vector.

$$\text{PE}_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$\text{PE}_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- $pos$: Position of the token in the sentence ($0, 1, 2, \dots, \text{seq\_len}-1$)
- $i$: Feature index inside the embedding vector ($0, 1, \dots, d_{model}/2 - 1$)
- $d_{model}$: Dimension of the embedding vector (e.g. $512$ or $128$)


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        
        # TODO: Create a positional encoding matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)
        
        # TODO: Create position tensor pos of shape (max_len, 1) -> [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # TODO: Compute the division term for sin/cos frequencies: 10000^(2i / d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # TODO: Apply sine to even indices (0, 2, 4, ...)
        # YOUR CODE HERE
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # TODO: Apply cosine to odd indices (1, 3, 5, ...)
        # YOUR CODE HERE
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: shape becomes (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # Register buffer so it is saved with model state_dict but not trained as a parameter
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (batch_size, seq_len, d_model)
        Add positional encodings to token embeddings
        """
        # TODO: Add positional encoding up to x's sequence length: x + self.pe[:, :x.size(1)]
        # YOUR CODE HERE
        x = x + self.pe[:, :x.size(1)]
        return x

# Quick Test for PositionalEncoding
pe_module = PositionalEncoding(d_model=16, max_len=10)
dummy_input = torch.zeros(1, 10, 16)
output = pe_module(dummy_input)
print("Positional Encoding Output Shape:", output.shape) # Expected: torch.Size([1, 10, 16])


## 2. Scaled Dot-Product Attention 🔍

Self-Attention allows tokens in a sentence to look at all other tokens and figure out which ones are most relevant to understanding their context.

For every input token embedding, we project it into **3 vectors**:
1. **Query ($Q$):** What this token is searching for.
2. **Key ($K$):** What this token contains (its index card).
3. **Value ($V$):** The actual information this token passes forward.

### Attention Equation:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

#### Steps:
1. **Dot Product Scores:** Compute raw similarity score between Query and Key: $S = Q K^T$. Shape: `(batch_size, seq_len, seq_len)`
2. **Scale:** Divide scores by $\sqrt{d_k}$ to prevent gradients from becoming too large.
3. **Softmax:** Convert scores into probability distribution (sum to 1 across columns).
4. **Weighted Sum:** Multiply probabilities by Value ($V$) to get the context-aware representation.


In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor = None):
    """
    Q shape: (batch_size, num_heads, seq_len, d_k)
    K shape: (batch_size, num_heads, seq_len, d_k)
    V shape: (batch_size, num_heads, seq_len, d_k)
    """
    d_k = Q.size(-1)
    
    # TODO Step 1 & 2: Compute scaled dot-product attention scores: (Q @ K^T) / sqrt(d_k)
    # Hint: Use K.transpose(-2, -1) to transpose the last two dimensions of K
    # YOUR CODE HERE
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply optional mask (used in decoder or padding)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    # TODO Step 3: Apply softmax across the last dimension to get attention weights
    # YOUR CODE HERE
    attn_weights = F.softmax(scores, dim=-1)
    
    # TODO Step 4: Multiply attention weights by Value tensor (V)
    # YOUR CODE HERE
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# Quick Test for Scaled Dot-Product Attention
batch_size, num_heads, seq_len, d_k = 2, 1, 4, 8
q_dummy = torch.randn(batch_size, num_heads, seq_len, d_k)
k_dummy = torch.randn(batch_size, num_heads, seq_len, d_k)
v_dummy = torch.randn(batch_size, num_heads, seq_len, d_k)

out, weights = scaled_dot_product_attention(q_dummy, k_dummy, v_dummy)
print("Attention Output Shape:", out.shape)          # Expected: [2, 1, 4, 8]
print("Attention Weights Shape:", weights.shape)    # Expected: [2, 1, 4, 4]


## 3. Multi-Head Attention (MHA) 👥

Instead of computing attention once (Single Head), **Multi-Head Attention** splits the embedding dimension into $h$ different "heads".

This allows the model to jointly attend to information from different representation subspaces at different positions!
- Head 1 might focus on syntactic relations (e.g., subject-verb agreement).
- Head 2 might focus on semantic pronouns (e.g., resolving "it" or "he").

### Steps:
1. Linear projections to get $Q, K, V$ from input $X$.
2. Split $Q, K, V$ into $h$ heads: Shape `(batch_size, num_heads, seq_len, d_k)`.
3. Compute **Scaled Dot-Product Attention** on each head.
4. Concatenate all head outputs back together.
5. Apply final linear projection ($W_O$).


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # TODO: Define Linear layers for Query, Key, Value projections
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # TODO: Define Output Linear projection
        self.out_linear = nn.Linear(d_model, d_model)
        
    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Reshape (batch_size, seq_len, d_model) -> (batch_size, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None):
        batch_size = q.size(0)
        
        # 1. Linear Projections & Split Heads
        # TODO: Project q, k, v using linear layers and call split_heads()
        # YOUR CODE HERE
        Q = self.split_heads(self.q_linear(q))
        K = self.split_heads(self.k_linear(k))
        V = self.split_heads(self.v_linear(v))
        
        # 2. Scaled Dot-Product Attention
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 3. Concatenate heads back together
        # (batch_size, num_heads, seq_len, d_k) -> (batch_size, seq_len, num_heads * d_k)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 4. Final Output Linear Projection
        # YOUR CODE HERE
        output = self.out_linear(attn_out)
        
        return output, attn_weights

# Quick Test for MultiHeadAttention
mha = MultiHeadAttention(d_model=64, num_heads=8)
x_dummy = torch.randn(2, 10, 64) # batch_size=2, seq_len=10, d_model=64
out, weights = mha(x_dummy, x_dummy, x_dummy)
print("MHA Output Shape:", out.shape)          # Expected: [2, 10, 64]
print("MHA Weights Shape:", weights.shape)    # Expected: [2, 8, 10, 10]


## 4. Position-Wise Feed-Forward Network & Layer Normalization ⚙️

Each Transformer layer consists of two main sub-layers:
1. **Multi-Head Self-Attention**
2. **Position-wise Feed-Forward Network (FFN)**

The Feed-Forward Network consists of two linear transformations with a ReLU/GELU activation in between:

$$\text{FFN}(x) = \max(0, x W_1 + b_1) W_2 + b_2$$

Additionally, **Residual Connections (Skip Connections)** and **Layer Normalization** are added around each sub-layer:

$$\text{LayerOutput} = \text{LayerNorm}(x + \text{SubLayer}(x))$$


In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super(PositionwiseFeedForward, self).__init__()
        # TODO: Define first linear layer from d_model -> d_ff
        self.fc1 = nn.Linear(d_model, d_ff)
        
        # TODO: Define second linear layer from d_ff -> d_model
        self.fc2 = nn.Linear(d_ff, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Apply fc1 -> ReLU -> Dropout -> fc2
        # YOUR CODE HERE
        x = self.fc2(self.dropout(F.relu(self.fc1(x))))
        return x

# Quick Test for FFN
ffn = PositionwiseFeedForward(d_model=64, d_ff=256)
x_test = torch.randn(2, 10, 64)
ffn_out = ffn(x_test)
print("FFN Output Shape:", ffn_out.shape) # Expected: [2, 10, 64]


## 5. Assembling the Transformer Encoder Block 🏗️

Now we combine **Multi-Head Attention**, **Feed-Forward Network**, **Residual Connections**, and **Layer Normalization** into a single **Transformer Encoder Layer**.

```
          Input Token Embeddings + Positional Encodings
                                │
                                ▼
                       ┌─────────────────┐
                       │ Multi-Head Attn │
                       └────────┬────────┘
                                │
                                ┼ ───► Residual Connection
                                ▼
                         LayerNorm(x)
                                │
                                ▼
                       ┌─────────────────┐
                       │  Feed-Forward   │
                       └────────┬────────┘
                                │
                                ┼ ───► Residual Connection
                                ▼
                         LayerNorm(x)
```


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super(TransformerEncoderLayer, self).__init__()
        
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # TODO Sub-layer 1: Self-Attention + Residual + LayerNorm
        # Hint: attn_out, _ = self.self_attn(x, x, x, mask)
        # YOUR CODE HERE
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # TODO Sub-layer 2: Feed-Forward + Residual + LayerNorm
        # YOUR CODE HERE
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x

# Quick Test for Encoder Layer
encoder_layer = TransformerEncoderLayer(d_model=64, num_heads=4, d_ff=128)
x_enc_test = torch.randn(2, 8, 64)
enc_out = encoder_layer(x_enc_test)
print("Encoder Layer Output Shape:", enc_out.shape) # Expected: [2, 8, 64]


## 6. Complete Transformer Model for Sequence Classification 🎯

Now let's build the full **Transformer Classifier** model:
1. **Token Embedding:** Converts token IDs to dense vectors `(batch_size, seq_len, d_model)`.
2. **Positional Encoding:** Adds spatial frequency signals to embeddings.
3. **Stacked Encoder Layers:** $N$ Transformer Encoder Blocks stacked on top of each other.
4. **Pooling & Classification Head:** Pools representations (e.g. mean pooling over sequence length) and passes through a Linear layer for binary sentiment classification (`0` or `1`).


In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, d_ff: int, num_layers: int, num_classes: int = 2, max_len: int = 500, dropout: float = 0.1):
        super(TransformerClassifier, self).__init__()
        
        # TODO: Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        
        # TODO: Stack N TransformerEncoderLayers using nn.ModuleList
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        
        # TODO: Linear Classifier Head (d_model -> num_classes)
        self.classifier = nn.Linear(d_model, num_classes)
        
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        x shape: (batch_size, seq_len)
        """
        # 1. Embedding + Positional Encoding
        # YOUR CODE HERE
        x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        # 2. Pass through stacked Encoder layers
        # YOUR CODE HERE
        for layer in self.layers:
            x = layer(x, mask)
            
        # 3. Global Mean Pooling over sequence dimension (dim=1)
        # YOUR CODE HERE
        x_pooled = x.mean(dim=1)
        
        # 4. Classification Output Logits
        # YOUR CODE HERE
        logits = self.classifier(x_pooled)
        return logits

# Quick Test for Full Transformer Classifier
vocab_size = 1000
model = TransformerClassifier(vocab_size=vocab_size, d_model=64, num_heads=4, d_ff=128, num_layers=2, num_classes=2)

sample_tokens = torch.randint(0, vocab_size, (4, 32)) # batch_size=4, seq_len=32
logits = model(sample_tokens)
print("Transformer Classifier Logits Output Shape:", logits.shape) # Expected: [4, 2]


## 7. Model Training & Synthetic Verification 🧪

To verify that our Transformer implementation learns properly, we construct a simple synthetic sequence task:
- If sequence contains token `10` -> Positive Sentiment (`Class 1`)
- Otherwise -> Negative Sentiment (`Class 0`)


In [ ]:
# Generate Synthetic Data
def generate_synthetic_data(num_samples=500, seq_len=20, vocab_size=1000):
    X = torch.randint(15, vocab_size, (num_samples, seq_len))
    y = torch.zeros(num_samples, dtype=torch.long)
    
    for i in range(num_samples):
        # Insert target token 10 at random position for Class 1
        if torch.rand(1).item() > 0.5:
            pos = torch.randint(0, seq_len, (1,)).item()
            X[i, pos] = 10
            y[i] = 1
            
    return X, y

X_train, y_train = generate_synthetic_data(num_samples=600, seq_len=20)
print("Training set shape:", X_train.shape, y_train.shape)

# Instantiate Model, Loss Function, and Optimizer
model = TransformerClassifier(vocab_size=1000, d_model=32, num_heads=2, d_ff=64, num_layers=2, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 10
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    preds = torch.argmax(outputs, dim=1)
    acc = (preds == y_train).float().mean()
    
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Accuracy: {acc.item()*100:.2f}%")


## 🎯 Summary & Next Steps

In this lab, you built a complete **Transformer Encoder Model from scratch**:
1. Added **Positional Encodings** to preserve sequence order.
2. Implemented **Scaled Dot-Product Attention** ($Q, K, V$).
3. Built **Multi-Head Attention** to capture diverse sub-space representations.
4. Combined components into a **Transformer Encoder Block** with Skip-Connections and LayerNorm.
5. Built a **Sequence Classifier** and verified model convergence.

### Recommended Readings & Further Exploration:
- 📖 [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
- 🎨 [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/)
- 💻 [The Annotated Transformer (Harvard NLP)](https://nlp.seas.harvard.edu/2018/04/03/attention.html)
